# SLIIT IT3051 - Data Mining Project (EV Battery PHM)
## Section A: Data Structure, Schema & Missingness Audit
### **Assigned Member: Person A**

---


### 0. Environment Setup & Configuration
Imports, dataset path, target/identifier definitions, train-test settings, and output-folder configuration used in my completed Python work.


In [ ]:
# IMPORTS AND CONFIGURATION

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer


# Repository dataset path
DATA_FILE = Path("ev battery_failure Dataset.csv")

if not DATA_FILE.exists():
    DATA_FILE = Path("../ev battery_failure Dataset.csv")


# Folder for Person A outputs
OUTPUT_DIR = Path("outputs")

if Path.cwd().name == "notebooks":
    OUTPUT_DIR = Path("../outputs")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TARGET = "battery_failure"

ID_COLUMNS = [
    "vehicle_id",
    "battery_serial"
]

POTENTIAL_LEAKAGE_COLUMNS = [
    "predicted_remaining_life_cycles"
]

TEST_SIZE = 0.20
RANDOM_STATE = 42


# Load dataset
df_raw = pd.read_csv(
    DATA_FILE,
    low_memory=False
)

df = df_raw.copy()

### Task 1: Data Ingestion & Dataset Dimensions
Load the raw EV battery dataset, preserve a copy, inspect its dimensions, and review sample records.


In [ ]:
# Load raw dataset
df_raw = pd.read_csv(DATA_FILE, low_memory=False)

# Create a copy of the raw dataset
df = df_raw.copy()

print(f"Rows          : {df.shape[0]:,}")
print(f"Columns       : {df.shape[1]:,}")
print(f"Dataset shape : {df.shape}")

print(df.head())
print(df.tail())



**Analysis & Viva Preparation Notes:**
- The raw dataset contains **20,000 rows and 70 columns**.
- `df_raw` is preserved as the original loaded dataset, while `df` is used as the working copy.
- Initial record inspection confirms that the dataset contains vehicle, battery, charging, driving, environmental, maintenance/fault, and target-related variables.


### Task 2: Column Names, Schema & Data Types Audit
Check column-name quality, duplicate column names, pandas data types, non-null counts, missing counts, missing percentages, and feature cardinality.


In [ ]:
# AUDIT COLUMN NAMES AND SCHEMA

print("Column names:")
print(df.columns.tolist())

# Check and remove accidental leading/trailing spaces in column names.
column_whitespace_issues = [
    column for column in df.columns
    if column != column.strip()
]

if column_whitespace_issues:
    print("\nColumns with accidental whitespace:")
    for column in column_whitespace_issues:
        print(repr(column))

    df.columns = [column.strip() for column in df.columns]
    print("Column-name whitespace was removed.")
else:
    print("\nNo column-name whitespace problems found.")

duplicate_column_names = df.columns[df.columns.duplicated()].tolist()
print("Duplicate column names:", duplicate_column_names)

print("\nDataFrame information:")
df.info()

print("\nPandas dtype summary:")
print(df.dtypes.value_counts())

schema_report = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (df.isna().mean().values * 100).round(2),
    "unique_values": df.nunique(dropna=True).values,
})

print("\nSchema report:")
print(schema_report.to_string(index=False))

schema_report.to_csv(
    OUTPUT_DIR / "01_schema_audit.csv",
    index=False
)




**Analysis & Viva Preparation Notes:**
- No leading/trailing whitespace problems were found in the column names.
- No duplicate column names were detected.
- Pandas identifies **59 `float64`, 10 string, and 1 `int64` columns**.
- The schema report provides a compact audit of datatype, non-null count, missing count, missing percentage, and unique-value count for every column.


### Task 3: Categorical Cleaning & Hidden Missing Values
Clean accidental whitespace in categorical values and convert recognized textual missing-value placeholders to actual `NaN` values.


In [ ]:
# CATEGORICAL CLEANING AND HIDDEN MISSING VALUES

missing_placeholders = {
    "",
    "na",
    "n/a",
    "null",
    "none",
    "?",
    "-",
    "--",
}

object_columns = df.select_dtypes(
    include=["object", "string"]
).columns.tolist()

hidden_missing_log = []

for column in object_columns:
    original = df[column].copy()

    # Remove only leading/trailing whitespace from text values.
    cleaned = original.astype("string").str.strip()

    whitespace_count = int(
        (
            original.notna()
            & (original.astype("string") != cleaned)
        ).sum()
    )

    # Convert pandas <NA> to standard np.nan for sklearn compatibility.
    df[column] = cleaned.astype(object)
    df[column] = df[column].where(pd.notna(df[column]), np.nan)

    # Detect hidden textual missing-value placeholders.
    normalized = df[column].astype("string").str.lower()
    placeholder_mask = normalized.isin(
        missing_placeholders
    ).fillna(False)

    placeholder_count = int(placeholder_mask.sum())

    if placeholder_count > 0:
        df.loc[placeholder_mask, column] = np.nan

    hidden_missing_log.append({
        "column": column,
        "whitespace_values_trimmed": whitespace_count,
        "placeholder_values_converted_to_nan": placeholder_count,
    })

hidden_missing_report = pd.DataFrame(hidden_missing_log)

print(hidden_missing_report.to_string(index=False))

hidden_missing_report.to_csv(
    OUTPUT_DIR / "02_text_cleaning_and_hidden_missing.csv",
    index=False
)



**Analysis & Viva Preparation Notes:**
- The audit checks common hidden missing representations such as blank strings, `NA`, `N/A`, `null`, `none`, `?`, `-`, and `--`.
- The current dataset did not contain hidden textual placeholders or categorical whitespace problems, but the handling step is retained so the workflow remains robust.


### Task 4: Missing-Value Audit
Measure missingness by column and by row, quantify overall missingness, evaluate the effect of complete-case deletion, and visualize missing-value percentages.


In [ ]:
# AUDIT MISSING-VALUES

missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean().values * 100
    ).round(2),
}).sort_values(
    "missing_percentage",
    ascending=False
).reset_index(drop=True)

print("Missing values by column:")
print(missing_summary.to_string(index=False))

missing_summary.to_csv(
    OUTPUT_DIR / "03_missing_value_summary.csv",
    index=False
)

# Dataset-level missingness.
total_cells = df.shape[0] * df.shape[1]
total_missing = int(df.isna().sum().sum())
overall_missing_percentage = total_missing / total_cells * 100
columns_with_missing = int((df.isna().sum() > 0).sum())

# Row-level missingness.
row_missing_count = df.isna().sum(axis=1)
rows_with_missing = int((row_missing_count > 0).sum())
complete_rows = int((row_missing_count == 0).sum())

print(f"\nTotal cells                        : {total_cells:,}")
print(f"Total missing cells                : {total_missing:,}")
print(f"Overall missing percentage         : {overall_missing_percentage:.2f}%")
print(
    f"Columns containing missing values  : "
    f"{columns_with_missing}/{df.shape[1]}"
)
print(
    f"Rows with >= 1 missing value       : "
    f"{rows_with_missing:,} "
    f"({rows_with_missing / len(df) * 100:.3f}%)"
)
print(
    f"Completely filled rows             : "
    f"{complete_rows:,} "
    f"({complete_rows / len(df) * 100:.3f}%)"
)
print(f"Median missing values per row      : {row_missing_count.median():.0f}")
print(f"Maximum missing values in one row  : {row_missing_count.max():.0f}")

# Demonstrate why complete-case deletion is unsuitable without modifying df.
rows_if_dropna_used = df.dropna().shape[0]

print(
    f"\nRows remaining if df.dropna() were used: "
    f"{rows_if_dropna_used:,}"
)
print(
    "Decision: complete-case deletion is NOT used because it would "
    "discard too much of the dataset."
)

# Visualize missing values 
missing_plot = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)
missing_plot = missing_plot[missing_plot > 0]

if not missing_plot.empty:
    plt.figure(figsize=(16, 7))
    missing_plot.plot(kind="bar")
    plt.title("Missing Values by Feature")
    plt.xlabel("Feature")
    plt.ylabel("Missing Values (%)")
    plt.xticks(rotation=90)
    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / "04_missing_values_by_feature.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()



**Analysis & Viva Preparation Notes:**
- The dataset contains **53,344 missing cells**, approximately **3.81%** of all cells.
- **67 of 70 columns** contain missing values.
- **18,663 rows (93.315%)** contain at least one missing value, while only **1,337 rows (6.685%)** are complete.
- Therefore, applying a global `dropna()` would cause excessive information loss, so complete-case deletion is not used as the general missing-value treatment.


### Task 5: Duplicate Audit & Handling
Check exact duplicate rows, duplicate identifiers, duplicates after identifiers are ignored, and retain the duplicate-removal step even when no duplicates are present.


In [ ]:
# DUPLICATE AUDIT AND HANDLING

# Exact duplicates.
exact_duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows before handling: {exact_duplicate_count}")

# Identifier-level duplicate checks.
for column in ID_COLUMNS:
    if column in df.columns:
        duplicated_id_rows = int(
            df[column].duplicated(keep=False).sum()
        )
        unique_id_values = int(
            df[column].nunique(dropna=True)
        )
        missing_id_values = int(
            df[column].isna().sum()
        )

        print(
            f"{column}: "
            f"unique={unique_id_values:,}, "
            f"rows participating in duplicate IDs={duplicated_id_rows:,}, "
            f"missing={missing_id_values:,}"
        )

# Check duplicate observations while ignoring record identifiers.
non_id_columns = [
    column for column in df.columns
    if column not in ID_COLUMNS
]

duplicates_without_ids = int(
    df.duplicated(
        subset=non_id_columns,
        keep="first"
    ).sum()
)

print(
    "Duplicates when identifier fields are ignored:",
    duplicates_without_ids
)

# HANDLE exact duplicates.
rows_before_duplicate_handling = len(df)

df = (
    df
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)

rows_after_duplicate_handling = len(df)

print(
    "Rows removed as exact duplicates:",
    rows_before_duplicate_handling - rows_after_duplicate_handling
)
print(
    "Exact duplicate rows after handling:",
    int(df.duplicated().sum())
)




**Analysis & Viva Preparation Notes:**
- **0 exact duplicate rows** were found.
- `vehicle_id` and `battery_serial` are unique and contain no missing values.
- No duplicate observations were found after ignoring the two identifier columns.
- The exact-duplicate handling step is still retained so the preprocessing workflow remains valid for future versions of the dataset.


### Task 6: Target Sanity Check
Validate the `battery_failure` target, check missing target labels, and confirm that the target contains only the expected binary classes.


In [ ]:
# TARGET SANITY CHECK

if TARGET not in df.columns:
    raise KeyError(
        f"Target column '{TARGET}' was not found."
    )

target_values = sorted(
    df[TARGET].dropna().unique().tolist()
)
missing_target_count = int(
    df[TARGET].isna().sum()
)

print("Target values:", target_values)
print("Missing target labels:", missing_target_count)


if missing_target_count > 0:
    print(
        f"Removing {missing_target_count} row(s) with missing target labels "
        "from supervised modelling data."
    )

    df = (
        df
        .dropna(subset=[TARGET])
        .reset_index(drop=True)
    )

# Confirm that the target is binary.
unexpected_target_values = set(df[TARGET].unique()) - {0, 1}

if unexpected_target_values:
    raise ValueError(
        f"Unexpected target values found: {unexpected_target_values}. "
        "Expected only 0 and 1."
    )



**Analysis & Viva Preparation Notes:**
- `battery_failure` contains only **0 and 1**, confirming a binary classification target.
- The target has **no missing values**.
- Target labels are never statistically imputed because doing so would create artificial class labels.


### Task 7: Clear Domain Checks & Invalid-Value Handling
Check defensible percentage, ratio, and count constraints. Clearly invalid values are converted to `NaN` so they can later be handled using training-only imputation.


In [ ]:
# CLEAR DOMAIN CHECKS AND HANDLING

# Percentage variables expected within [0, 100].
percentage_columns = [
    "battery_health_percent",
    "state_of_charge",
    "state_of_health",
    "capacity_loss_percent",
    "charge_efficiency",
    "discharge_efficiency",
    "humidity",
]

# Ratio variables expected within [0, 1].
ratio_columns = [
    "fast_charge_ratio",
    "slow_charge_ratio",
    "overnight_charging_ratio",
    "home_charging_ratio",
    "highway_driving_ratio",
    "city_driving_ratio",
]

# Count variables cannot logically be negative.
count_columns = [
    "cycle_count",
    "charging_cycles_last_month",
    "charging_interruptions",
    "overcharge_events",
    "firmware_updates",
    "previous_faults",
    "sensor_fault_count",
    "BMS_warning_count",
    "abnormal_voltage_events",
]

invalid_value_log = []

for column in percentage_columns:
    if column in df.columns:
        invalid_mask = (
            (df[column] < 0)
            | (df[column] > 100)
        )
        invalid_count = int(invalid_mask.sum())

        invalid_value_log.append({
            "column": column,
            "valid_rule": "0 <= value <= 100",
            "invalid_count": invalid_count,
            "handling": (
                "Invalid values converted to NaN for later "
                "training-only imputation"
            ),
        })

        if invalid_count > 0:
            df.loc[invalid_mask, column] = np.nan

for column in ratio_columns:
    if column in df.columns:
        invalid_mask = (
            (df[column] < 0)
            | (df[column] > 1)
        )
        invalid_count = int(invalid_mask.sum())

        invalid_value_log.append({
            "column": column,
            "valid_rule": "0 <= value <= 1",
            "invalid_count": invalid_count,
            "handling": (
                "Invalid values converted to NaN for later "
                "training-only imputation"
            ),
        })

        if invalid_count > 0:
            df.loc[invalid_mask, column] = np.nan

for column in count_columns:
    if column in df.columns:
        invalid_mask = df[column] < 0
        invalid_count = int(invalid_mask.sum())

        invalid_value_log.append({
            "column": column,
            "valid_rule": "value >= 0",
            "invalid_count": invalid_count,
            "handling": (
                "Invalid values converted to NaN for later "
                "training-only imputation"
            ),
        })

        if invalid_count > 0:
            df.loc[invalid_mask, column] = np.nan

invalid_value_report = pd.DataFrame(invalid_value_log)

print(invalid_value_report.to_string(index=False))

invalid_value_report.to_csv(
    OUTPUT_DIR / "05_invalid_value_handling.csv",
    index=False
)



**Analysis & Viva Preparation Notes:**
- No invalid values were detected under the implemented percentage (`0–100`), ratio (`0–1`), or non-negative count rules.
- Converting clearly impossible values to `NaN` is safer than inventing a replacement because the actual measurement is unknown.


### Task 8: Cross-Variable Sanity Checks
Check logical consistency between related measurements without automatically modifying values when the exact variable definition is uncertain.


In [ ]:
# CROSS-VARIABLE SANITY CHECKS

sanity_check_rows = []

if {
    "remaining_capacity",
    "battery_capacity_kwh",
}.issubset(df.columns):
    suspicious_count = int(
        (
            df["remaining_capacity"]
            > df["battery_capacity_kwh"]
        ).sum()
    )

    sanity_check_rows.append({
        "check": "remaining_capacity <= battery_capacity_kwh",
        "suspicious_rows": suspicious_count,
        "action": "Flag for investigation if non-zero",
    })

if {
    "cell_temperature_max",
    "cell_temperature_avg",
}.issubset(df.columns):
    suspicious_count = int(
        (
            df["cell_temperature_max"]
            < df["cell_temperature_avg"]
        ).sum()
    )

    sanity_check_rows.append({
        "check": "cell_temperature_max >= cell_temperature_avg",
        "suspicious_rows": suspicious_count,
        "action": "Flag for investigation if non-zero",
    })

if {
    "minimum_temperature",
    "average_ambient_temperature",
    "maximum_temperature",
}.issubset(df.columns):
    invalid_temperature_order = (
        (
            df["minimum_temperature"]
            > df["average_ambient_temperature"]
        )
        |
        (
            df["average_ambient_temperature"]
            > df["maximum_temperature"]
        )
    )

    sanity_check_rows.append({
        "check": (
            "minimum_temperature <= average_ambient_temperature "
            "<= maximum_temperature"
        ),
        "suspicious_rows": int(invalid_temperature_order.sum()),
        "action": "Flag for investigation if non-zero",
    })

# Keep this as a flag only because the exact definitions must be confirmed
# before assuming city and highway ratios are mutually exclusive shares.
if {
    "city_driving_ratio",
    "highway_driving_ratio",
}.issubset(df.columns):
    complete_ratio_rows = (
        df["city_driving_ratio"].notna()
        & df["highway_driving_ratio"].notna()
    )

    combined_ratio = (
        df.loc[complete_ratio_rows, "city_driving_ratio"]
        + df.loc[complete_ratio_rows, "highway_driving_ratio"]
    )

    sanity_check_rows.append({
        "check": "city_driving_ratio + highway_driving_ratio > 1",
        "suspicious_rows": int((combined_ratio > 1).sum()),
        "action": (
            "Flag only; do not modify until variable definitions "
            "confirm the expected relationship"
        ),
    })

sanity_report = pd.DataFrame(sanity_check_rows)

print(sanity_report.to_string(index=False))

sanity_report.to_csv(
    OUTPUT_DIR / "06_cross_variable_sanity_checks.csv",
    index=False
)



**Analysis & Viva Preparation Notes:**
- No violations were found for remaining capacity versus total battery capacity, maximum versus average cell temperature, or minimum/average/maximum ambient-temperature ordering.
- **9,254 rows** have `city_driving_ratio + highway_driving_ratio > 1`.
- This is flagged rather than corrected because the dataset definition must first confirm that those two ratios are mutually exclusive shares.


### Task 9: Target Class Distribution
Inspect the balance of the binary `battery_failure` target.


In [ ]:
# TARGET CLASS DISTRIBUTION

target_distribution = pd.DataFrame({
    "count": df[TARGET].value_counts().sort_index(),
    "percentage": (
        df[TARGET]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    ),
}).reset_index()

target_distribution.columns = [
    "class",
    "count",
    "percentage"
]

print(target_distribution.to_string(index=False))

target_distribution.to_csv(
    OUTPUT_DIR / "07_target_class_distribution.csv",
    index=False
)



**Analysis & Viva Preparation Notes:**
- Class `0` contains **18,616 records (93.08%)**.
- Class `1` contains **1,384 records (6.92%)**.
- The target is therefore imbalanced. The distribution must be considered during train-test splitting and later model evaluation.


### Task 10: Prepare Modelling Features & Leakage Review
Exclude record identifiers from predictive inputs and temporarily exclude the precomputed `predicted_remaining_life_cycles` feature until its source is verified.


In [ ]:
# PREPARE MODEL FEATURES

# Keeped identifiers in the master dataframe for traceability, but removed them from the predictive feature matrix.

model_exclusions = [
    column for column in ID_COLUMNS
    if column in df.columns
]

# Exclude this predicted feature until its source is verified to avoid possible data leakage.

for column in POTENTIAL_LEAKAGE_COLUMNS:
    if column in df.columns:
        model_exclusions.append(column)

print("Excluded from modelling:", model_exclusions)

X = df.drop(
    columns=model_exclusions + [TARGET]
)
y = df[TARGET].copy()

# Imputation strategies.

numeric_columns = (
    X.select_dtypes(include=[np.number])
    .columns
    .tolist()
)

categorical_columns = (
    X.select_dtypes(
        include=["object", "string", "category"]
    )
    .columns
    .tolist()
)

print(f"Total modelling features : {X.shape[1]}")
print(f"Numerical features       : {len(numeric_columns)}")
print(f"Categorical features     : {len(categorical_columns)}")
print(
    f"Missing feature cells    : "
    f"{int(X.isna().sum().sum()):,}"
)




**Analysis & Viva Preparation Notes:**
- `vehicle_id` and `battery_serial` are retained in the master data for traceability but excluded from model inputs.
- `predicted_remaining_life_cycles` is temporarily excluded because its calculation method is not yet verified and it may introduce leakage.
- After these exclusions, the modelling feature set contains **66 features: 58 numerical and 8 categorical**.


### Task 11: Train/Test Split Before Statistical Imputation
Split the data before learning medians or any training-derived relationships so test information does not leak into preprocessing.


In [ ]:
# TRAIN/TEST SPLIT BEFORE STATISTICAL IMPUTATION


# Stratification keeps approximately the same target-class proportions in both
# training and testing data.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")

print("\nTraining target distribution (%):")
print(
    (
        y_train
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

print("\nTesting target distribution (%):")
print(
    (
        y_test
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)




**Analysis & Viva Preparation Notes:**
- The dataset is split **80% training / 20% testing**, giving **16,000 training rows and 4,000 test rows**.
- `stratify=y` keeps the failure/non-failure class proportions approximately equal in both sets.
- The split is performed before statistical imputation to prevent test-set information from influencing training preprocessing.


### Task 12: Type-Specific & Relationship-Aware Missing-Value Handling
Use known row-level relationships where they are strong and defensible, then use training-only fallback imputation for unresolved values.


In [ ]:
# TYPE-SPECIFIC MISSING-VALUE IMPUTATION

# Preserve the original split for before/after comparison.
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()




#### 12.1 Battery Health Imputation
Use the relationship `battery_health_percent = 100 - capacity_loss_percent` when capacity loss is available.


In [ ]:
# IMPUTATION: BATTERY HEALTH

# battery_health_percent = 100 - capacity_loss_percent

for dataset_name, dataset in [
    ("X_train", X_train_imputed),
    ("X_test", X_test_imputed),
]:
    if {
        "battery_health_percent",
        "capacity_loss_percent",
    }.issubset(dataset.columns):

        missing_health_mask = (
            dataset["battery_health_percent"].isna()
            & dataset["capacity_loss_percent"].notna()
        )

        filled_count = int(missing_health_mask.sum())

        dataset.loc[
            missing_health_mask,
            "battery_health_percent"
        ] = (
            100
            - dataset.loc[
                missing_health_mask,
                "capacity_loss_percent"
            ]
        )

        print(
            f"{dataset_name}: battery_health_percent filled from "
            f"capacity_loss_percent = {filled_count}"
        )




#### 12.2 Vehicle Brand Imputation
Learn `vehicle_model → vehicle_brand` relationships from the training data only and infer a brand only when the model maps unambiguously to one brand.


In [ ]:
# IMPUTATION: VEHICLE BRAND

# Learn vehicle_model -> vehicle_brand relationships from TRAINING data only.
# Use a model to infer the brand only when that model maps to exactly one brand in the training data. Ambiguous or unknown cases are left missing for now.

if {
    "vehicle_model",
    "vehicle_brand",
}.issubset(X_train.columns):

    model_brand_source = (
        X_train[
            ["vehicle_model", "vehicle_brand"]
        ]
        .dropna()
    )

    brand_count_per_model = (
        model_brand_source
        .groupby("vehicle_model")["vehicle_brand"]
        .nunique()
    )

    unambiguous_models = (
        brand_count_per_model[
            brand_count_per_model == 1
        ].index
    )

    model_to_brand = (
        model_brand_source[
            model_brand_source["vehicle_model"].isin(
                unambiguous_models
            )
        ]
        .drop_duplicates(subset=["vehicle_model"])
        .set_index("vehicle_model")["vehicle_brand"]
        .to_dict()
    )

    for dataset_name, dataset in [
        ("X_train", X_train_imputed),
        ("X_test", X_test_imputed),
    ]:
        missing_brand_mask = (
            dataset["vehicle_brand"].isna()
        )

        inferred_brand = (
            dataset.loc[
                missing_brand_mask,
                "vehicle_model"
            ]
            .map(model_to_brand)
        )

        inferred_indexes = (
            inferred_brand[
                inferred_brand.notna()
            ].index
        )

        dataset.loc[
            inferred_indexes,
            "vehicle_brand"
        ] = inferred_brand.loc[
            inferred_indexes
        ]

        print(
            f"{dataset_name}: vehicle_brand filled from "
            f"vehicle_model = {len(inferred_indexes)}"
        )




#### 12.3 Numerical Fallback Imputation
Use the **training median** for remaining numerical/count features because it is less sensitive to skewness and extreme values than the mean.


In [ ]:
# NUMERICAL IMPUTATION -> TRAINING MEDIAN

relationship_numeric_columns = [
    "capacity_loss_percent",
    "remaining_capacity",
]

numeric_columns_for_median = [
    column
    for column in numeric_columns
    if column not in relationship_numeric_columns
]

# Median is used for remaining numerical missing values because it is less sensitive to skewness and extreme observations than the mean.
numeric_imputer = SimpleImputer(
    strategy="median"
)

# FIT numerical imputation values on TRAINING data only.
if numeric_columns_for_median:
    X_train_imputed[
        numeric_columns_for_median
    ] = numeric_imputer.fit_transform(
        X_train_imputed[
            numeric_columns_for_median
        ]
    )

    # Apply the SAME learned medians to the test set.
    X_test_imputed[
        numeric_columns_for_median
    ] = numeric_imputer.transform(
        X_test_imputed[
            numeric_columns_for_median
        ]
    )




#### 12.4 Capacity Loss Imputation
Calculate missing `capacity_loss_percent` values from the battery-health relationship.


In [ ]:
# IMPUTATION: CAPACITY LOSS

# capacity_loss_percent = 100 - battery_health_percent

for dataset_name, dataset in [
    ("X_train", X_train_imputed),
    ("X_test", X_test_imputed),
]:
    if {
        "capacity_loss_percent",
        "battery_health_percent",
    }.issubset(dataset.columns):

        missing_capacity_loss_mask = (
            dataset["capacity_loss_percent"].isna()
        )

        filled_count = int(
            missing_capacity_loss_mask.sum()
        )

        dataset.loc[
            missing_capacity_loss_mask,
            "capacity_loss_percent"
        ] = (
            100
            - dataset.loc[
                missing_capacity_loss_mask,
                "battery_health_percent"
            ]
        )

        print(
            f"{dataset_name}: capacity_loss_percent filled from "
            f"battery_health_percent = {filled_count}"
        )




#### 12.5 Remaining Capacity Imputation
Calculate missing remaining capacity using `battery_capacity_kwh × battery_health_percent / 100`.


In [ ]:
# IMPUTATION: REMAINING CAPACITY

# remaining_capacity is derived approximately as: battery_capacity_kwh * battery_health_percent / 100

for dataset_name, dataset in [
    ("X_train", X_train_imputed),
    ("X_test", X_test_imputed),
]:
    if {
        "remaining_capacity",
        "battery_capacity_kwh",
        "battery_health_percent",
    }.issubset(dataset.columns):

        missing_remaining_capacity_mask = (
            dataset["remaining_capacity"].isna()
        )

        filled_count = int(
            missing_remaining_capacity_mask.sum()
        )

        dataset.loc[
            missing_remaining_capacity_mask,
            "remaining_capacity"
        ] = (
            dataset.loc[
                missing_remaining_capacity_mask,
                "battery_capacity_kwh"
            ]
            * dataset.loc[
                missing_remaining_capacity_mask,
                "battery_health_percent"
            ]
            / 100
        )

        print(
            f"{dataset_name}: remaining_capacity filled from "
            f"battery capacity and battery health = {filled_count}"
        )




#### 12.6 Categorical Fallback & Imputation Verification
Use an explicit `"Missing"` category for unresolved categorical values rather than inventing the globally most frequent category, then verify that no missing model-feature values remain.


In [ ]:
# CATEGORICAL IMPUTATION -> "Missing"

#unresolved brand is also handled as "Missing".
categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Missing"
)

# FIT on TRAINING data and apply the same rule to the test set.
if categorical_columns:
    X_train_imputed[
        categorical_columns
    ] = categorical_imputer.fit_transform(
        X_train_imputed[
            categorical_columns
        ]
    )

    X_test_imputed[
        categorical_columns
    ] = categorical_imputer.transform(
        X_test_imputed[
            categorical_columns
        ]
    )


print("Missing cells before imputation:")
print(f"  X_train: {int(X_train.isna().sum().sum()):,}")
print(f"  X_test : {int(X_test.isna().sum().sum()):,}")

print("\nMissing cells after imputation:")
print(f"  X_train: {int(X_train_imputed.isna().sum().sum()):,}")
print(f"  X_test : {int(X_test_imputed.isna().sum().sum()):,}")

# Verify that the imputation step actually completed successfully.
if X_train_imputed.isna().any().any():
    remaining_columns = (
        X_train_imputed
        .columns[X_train_imputed.isna().any()]
        .tolist()
    )

    raise ValueError(
        "Missing values remain in X_train after imputation: "
        f"{remaining_columns}"
    )

if X_test_imputed.isna().any().any():
    remaining_columns = (
        X_test_imputed
        .columns[X_test_imputed.isna().any()]
        .tolist()
    )

    raise ValueError(
        "Missing values remain in X_test after imputation: "
        f"{remaining_columns}"
    )

print(
    "\nImputation verification passed: "
    "no missing model-feature values remain."
)




**Analysis & Viva Preparation Notes:**
- `battery_health_percent` is first reconstructed from `capacity_loss_percent` where possible; unresolved numerical values use training medians.
- `vehicle_brand` is inferred from `vehicle_model` only when the training data gives an unambiguous mapping; unresolved categorical values are labelled `"Missing"`.
- `capacity_loss_percent` is derived as `100 - battery_health_percent`.
- `remaining_capacity` is derived as `battery_capacity_kwh × battery_health_percent / 100`.
- In the completed run, relationship-based handling filled many values before fallback imputation, and the final verification showed **0 missing values in both `X_train` and `X_test`**.


### Task 13: Save Training Imputation Statistics
Record the training-derived medians and the relationship/categorical strategies used so preprocessing decisions remain auditable.


In [ ]:
# SAVE TRAINING IMPUTATION STATISTICS

imputation_rows = []

if numeric_columns_for_median:
    for column, median_value in zip(
        numeric_columns_for_median,
        numeric_imputer.statistics_
    ):
        if column == "battery_health_percent":
            strategy = (
                "capacity-loss relationship first; "
                "training median fallback"
            )
        else:
            strategy = "training median"

        imputation_rows.append({
            "column": column,
            "feature_type": "numeric/count",
            "strategy": strategy,
            "imputation_value": median_value,
        })

# Record the relationship-based numerical strategies separately.
if "capacity_loss_percent" in X.columns:
    imputation_rows.append({
        "column": "capacity_loss_percent",
        "feature_type": "numeric/derived",
        "strategy": "100 - battery_health_percent",
        "imputation_value": "row-level relationship",
    })

if "remaining_capacity" in X.columns:
    imputation_rows.append({
        "column": "remaining_capacity",
        "feature_type": "numeric/derived",
        "strategy": (
            "battery_capacity_kwh * "
            "battery_health_percent / 100"
        ),
        "imputation_value": "row-level relationship",
    })

if categorical_columns:
    for column in categorical_columns:
        if column == "vehicle_brand":
            strategy = (
                "training vehicle_model->brand mapping; "
                "fallback Missing"
            )
        else:
            strategy = "constant Missing category"

        imputation_rows.append({
            "column": column,
            "feature_type": "categorical",
            "strategy": strategy,
            "imputation_value": "Missing",
        })

imputation_statistics = pd.DataFrame(
    imputation_rows
)

print(
    imputation_statistics.to_string(
        index=False
    )
)

imputation_statistics.to_csv(
    OUTPUT_DIR / "08_training_imputation_statistics.csv",
    index=False
)




**Analysis & Viva Preparation Notes:**
- Saving the learned imputation values makes the preprocessing reproducible and provides clear evidence that numerical fallback values were learned from training data only.
- Relationship-based and categorical strategies are also documented separately.


### Task 14: Save Practical Outputs
Save the cleaned pre-imputation dataset and the exact imputed train/test feature and target sets for team handoff.


In [ ]:
# SAVE PRACTICAL OUTPUTS

df.to_csv(
    OUTPUT_DIR / "09_cleaned_before_statistical_imputation.csv",
    index=False
)

# Save the exact split prepared with training-fitted imputation.
X_train_imputed.to_csv(
    OUTPUT_DIR / "10_X_train_imputed.csv",
    index=False
)

X_test_imputed.to_csv(
    OUTPUT_DIR / "11_X_test_imputed.csv",
    index=False
)

y_train.to_frame(name=TARGET).to_csv(
    OUTPUT_DIR / "12_y_train.csv",
    index=False
)

y_test.to_frame(name=TARGET).to_csv(
    OUTPUT_DIR / "13_y_test.csv",
    index=False
)

print(f"Outputs saved to: {OUTPUT_DIR}")




**Analysis & Viva Preparation Notes:**
- The pre-imputation cleaned dataset is retained separately from the statistically imputed train/test sets.
- Saving the exact split allows later team members to work from the same prepared data if the team decides to use this split.


### Task 15: Final Audit Summary
Create a concise summary of the main Person A audit and preprocessing outcomes.


In [ ]:
# FINAL SUMMARY

summary = pd.DataFrame(
    [
        ["Original rows", len(df_raw)],
        ["Original columns", df_raw.shape[1]],
        ["Exact duplicates originally found", exact_duplicate_count],
        ["Rows after duplicate handling", len(df)],
        ["Total missing cells at audit stage", total_missing],
        ["Overall missing percentage", round(overall_missing_percentage, 4)],
        ["Columns with missing values", columns_with_missing],
        ["Rows with at least one missing value", rows_with_missing],
        ["Complete rows", complete_rows],
        ["Numerical modelling features", len(numeric_columns)],
        ["Categorical modelling features", len(categorical_columns)],
        [
            "Missing X_train values after imputation",
            int(X_train_imputed.isna().sum().sum())
        ],
        [
            "Missing X_test values after imputation",
            int(X_test_imputed.isna().sum().sum())
        ],
    ],
    columns=["metric", "value"]
)

print(summary.to_string(index=False))

summary.to_csv(
    OUTPUT_DIR / "14_final_summary.csv",
    index=False
)






**Analysis & Viva Preparation Notes:**
- Original dataset: **20,000 rows × 70 columns**.
- Exact duplicates found: **0**.
- Overall missingness: approximately **3.81%**, spread across **67 columns**.
- Rows containing at least one missing value: **18,663**.
- Final modelling inputs after exclusions: **58 numerical + 8 categorical features**.
- Missing values after the completed training/test imputation workflow: **0 in training and 0 in testing**.
